In [11]:
!pip install librosa


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\Lenovo\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [12]:
import http.client

conn = http.client.HTTPSConnection("api.reccobeats.com")
payload = ''
headers = {
  'Accept': 'application/json'
}
conn.request("GET", "/v1/track/878dadea-33c5-4c08-bdb9-e2b117475a99/audio-features", payload, headers)
res = conn.getresponse()
data = res.read()
print(data.decode("utf-8"))

{"id":"878dadea-33c5-4c08-bdb9-e2b117475a99","href":"https://open.spotify.com/track/00vJzaoxM3Eja1doBUhX0P","isrc":"USCJY1231021","acousticness":0.0448,"danceability":0.463,"energy":0.605,"instrumentalness":3.34E-4,"key":0,"liveness":0.128,"loudness":-7.803,"mode":1,"speechiness":0.0293,"tempo":186.113,"valence":0.317}


In [ ]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
import requests
import time
import csv
import pandas as pd
from tqdm import tqdm
from spotipy.exceptions import SpotifyException
import re
import os

MAX_TRACKS_TOTAL = 100000
ALBUM_TRACKS_LIMIT = 50
RECCOBEATS_BATCH_SIZE = 40
RECCOBEATS_TIMEOUT = 30
CLIENT_ID =  os.getenv("SPOTIFY_CLIENT_ID")
CLIENT_SECRET = os.getenv("SPOTIFY_CLIENT_SECRET")

credentials = {
    'client_id': CLIENT_ID,
    'client_secret': CLIENT_SECRET
}

sp = spotipy.Spotify(
    client_credentials_manager=SpotifyClientCredentials(
        client_id=credentials["client_id"],
        client_secret=credentials["client_secret"]
    )
)

AUDIO_FEATURES_FILE = "audio_features.csv"

print("="*70)
print("🎵 CONTINUOUS AUDIO FEATURES SCRAPER")
print("="*70)

print("\n📀 Loading existing tracks...")

try:
    existing_df = pd.read_csv(AUDIO_FEATURES_FILE)
    existing_track_ids = set(existing_df['id'].dropna().tolist())
    print(f"✅ Loaded {len(existing_track_ids)} existing tracks")
except Exception as e:
    print(f"No existing file or error: {e}")
    existing_track_ids = set()
    existing_df = None


playlist_uri = "spotify:playlist:6yPiKpy7evrwvZodByKvM9"
playlist_id = playlist_uri.split(":")[2]

artist_ids = set()
offset = 0
print("\n▶ Collecting artists from playlist...")

while True:
    results = sp.playlist_items(playlist_id, limit=100, offset=offset)
    items = results.get("items", [])
    if not items:
        break

    for item in items:
        track = item.get("track")
        if track:
            for artist in track.get("artists", []):
                artist_ids.add(artist["id"])

    offset += len(items)

artist_ids = list(artist_ids)
print(f"✓ Artists collected: {len(artist_ids)}")


all_track_ids = set(existing_track_ids)  # Start with existing
seen_albums = set()

print("\n▶ Collecting tracks from artists...")

for ai, artist_id in enumerate(tqdm(artist_ids, desc="Artists", total=len(artist_ids))):
    offset = 0

    while True:
        try:
            results = sp.artist_albums(
                artist_id, album_type="album,single", limit=50, offset=offset
            )
        except SpotifyException as e:
            print(f"\n  ✗ Error: {e}")
            break
        except Exception as e:
            print(f"\n  ✗ Error: {e}")
            break

        albums = results.get("items", [])
        if not albums:
            break

        for album in albums:
            album_id = album["id"]
            if album_id in seen_albums:
                continue
            seen_albums.add(album_id)

            try:
                track_results = sp.album_tracks(album_id, limit=ALBUM_TRACKS_LIMIT)
                tracks = track_results.get("items", [])

                while track_results.get("next"):
                    track_results = sp.next(track_results)
                    tracks.extend(track_results.get("items", []))

                for t in tracks:
                    tid = t.get("id")
                    if tid:
                        all_track_ids.add(tid)

                    if len(all_track_ids) >= MAX_TRACKS_TOTAL:
                        break

            except SpotifyException as e:
                print(f"    ✗ track error: {e}")
                time.sleep(5)

            if len(all_track_ids) >= MAX_TRACKS_TOTAL:
                break

        if len(all_track_ids) >= MAX_TRACKS_TOTAL:
            break

        offset += len(albums)
        time.sleep(0.1)

    if len(all_track_ids) >= MAX_TRACKS_TOTAL:
        break

print(f"\n✓ Track collection complete!")
print(f"  Total unique tracks: {len(all_track_ids)}")
print(f"  New tracks to fetch: {len(all_track_ids) - len(existing_track_ids)}")


new_track_ids = all_track_ids - existing_track_ids

if new_track_ids:
    print(f"\n▶ Fetching ReccoBeats features for {len(new_track_ids)} new tracks...")
    
    BASE_URL = "https://api.reccobeats.com/v1/audio-features"
    
    fieldnames = [
        "id", "trackTitle", "artists", "durationMs",
        "acousticness", "danceability", "energy",
        "instrumentalness", "key", "liveness",
        "loudness", "mode", "speechiness", "tempo",
        "valence", "href"
    ]
    
    # Append mode
    mode = 'a' if existing_df is not None else 'w'
    
    with open(AUDIO_FEATURES_FILE, mode, newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        
        if mode == 'w':
            writer.writeheader()
        
        track_list = list(new_track_ids)
        batches = [track_list[i:i + RECCOBEATS_BATCH_SIZE] for i in range(0, len(track_list), RECCOBEATS_BATCH_SIZE)]
        
        for i, batch in enumerate(tqdm(batches, desc="Feature batches")):
            ids_param = ",".join(batch)
            url = f"{BASE_URL}?ids={ids_param}"
            
            try:
                res = requests.get(url, headers={"Accept": "application/json"}, timeout=RECCOBEATS_TIMEOUT)
            except requests.exceptions.Timeout:
                print(f"  ✗ timeout on batch {i+1}, skipping")
                continue
            
            if res.status_code == 200:
                data = res.json()
                for track in data.get("content", []):
                    writer.writerow({
                        "id": track.get("id"),
                        "trackTitle": track.get("trackTitle"),
                        "artists": ";".join(a["name"] for a in track.get("artists", [])),
                        "durationMs": track.get("durationMs"),
                        "acousticness": track.get("acousticness"),
                        "danceability": track.get("danceability"),
                        "energy": track.get("energy"),
                        "instrumentalness": track.get("instrumentalness"),
                        "key": track.get("key"),
                        "liveness": track.get("liveness"),
                        "loudness": track.get("loudness"),
                        "mode": track.get("mode"),
                        "speechiness": track.get("speechiness"),
                        "tempo": track.get("tempo"),
                        "valence": track.get("valence"),
                        "href": track.get("href")
                    })
            elif res.status_code == 429:
                print("  ⚠ rate limit hit — sleeping 10s")
                time.sleep(10)
            else:
                print(f"  ✗ error {res.status_code}: {res.text[:100]}")
                time.sleep(1)
    
    print(f"✓ Appended {len(new_track_ids)} new tracks to {AUDIO_FEATURES_FILE}")
else:
    print("\n✅ No new tracks to fetch!")


print("\n" + "="*60)
print("🎵 Updating Spotify track names and artists for new tracks")
print("="*60)

# Reload the updated file
df = pd.read_csv(AUDIO_FEATURES_FILE, on_bad_lines='skip')
print(f"📀 Loaded {len(df)} total tracks")

# Function to extract Spotify ID from href
def extract_spotify_id(url):
    if pd.isna(url):
        return None
    match = re.search(r'track/([a-zA-Z0-9]{22})', str(url))
    return match.group(1) if match else None

# Add spotify_id column if not exists
if 'spotify_id' not in df.columns:
    df['spotify_id'] = df['href'].apply(extract_spotify_id)

# Find tracks missing Spotify names
if 'trackTitle' not in df.columns:
    df['trackTitle'] = None
if 'artists' not in df.columns:
    df['artists'] = None

missing_names = df[df['trackTitle'].isna() & df['spotify_id'].notna()]
print(f"\n📊 Need to fetch Spotify names for {len(missing_names)} tracks")

if len(missing_names) > 0:
    spotify_ids_to_fetch = missing_names['spotify_id'].tolist()
    batch_size = 50
    fetched_count = 0
    
    print(f"\n🔍 Fetching Spotify track names...")
    
    for i in tqdm(range(0, len(spotify_ids_to_fetch), batch_size), desc="Fetching"):
        batch_ids = spotify_ids_to_fetch[i:i+batch_size]
        
        try:
            tracks_info = sp.tracks(batch_ids)
            
            for track_data in tracks_info['tracks']:
                if track_data:
                    spotify_id = track_data['id']
                    track_name = track_data['name']
                    artists = ", ".join([artist['name'] for artist in track_data['artists']])
                    
                    # Update dataframe
                    mask = df['spotify_id'] == spotify_id
                    if mask.any():
                        idx = df[mask].index[0]
                        df.loc[idx, 'trackTitle'] = track_name
                        df.loc[idx, 'artists'] = artists
                        fetched_count += 1
                        
        except Exception as e:
            print(f"  Error: {e}")
        
        time.sleep(0.2)
    
    print(f"✅ Fetched {fetched_count} Spotify track names")
    
    # Fill any remaining with placeholders
    df['trackTitle'] = df['trackTitle'].fillna('Unknown_Track')
    df['artists'] = df['artists'].fillna('Unknown_Artist')
    
    # Save updated file
    df.to_csv(AUDIO_FEATURES_FILE, index=False)
    print(f"✅ Saved updated file with Spotify names")
else:
    print("✅ All tracks already have Spotify names!")


print("\n" + "="*60)
print("📊 FINAL SUMMARY")
print("="*60)

tracks_with_names = df[df['trackTitle'] != 'Unknown_Track']

print(f"\nTotal tracks: {len(df)}")
print(f"Tracks with names: {len(tracks_with_names)}")
print(f"Tracks with Spotify IDs: {df['spotify_id'].notna().sum()}")

print(f"\n🎵 Sample tracks:")
for i in range(min(5, len(tracks_with_names))):
    track = tracks_with_names.iloc[i]
    print(f"  {track['trackTitle'][:50]} - {track['artists'][:40]}")
    print(f"    Valence: {track['valence']:.3f}, Energy: {track['energy']:.3f}")

print("\n" + "="*60)
print("✅ COMPLETE!")
print("="*60)

🎵 CONTINUOUS AUDIO FEATURES SCRAPER

📀 Loading existing tracks...
✅ Loaded 28928 existing tracks

▶ Collecting artists from playlist...
✓ Artists collected: 4200

▶ Collecting tracks from artists...


Artists:   0%|          | 0/4200 [00:00<?, ?it/s]

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,

🎵 FIXING AUDIO FEATURES - ADDING TRACK NAMES & ARTISTS

📀 Loading audio_features.csv...
✅ Loaded 5014 tracks
Columns: ['id', 'trackTitle', 'artists', 'durationMs', 'acousticness', 'danceability', 'energy', 'instrumentalness', 'key', 'liveness', 'loudness', 'mode', 'speechiness', 'tempo', 'valence', 'href']

🔍 Extracting Spotify IDs from href URLs...
✅ Extracted 5014 Spotify IDs out of 5014 tracks

Sample of extracted IDs:
  https://open.spotify.com/track/0IxxqsYBcCHEQ1HqLYJ... → 0IxxqsYBcCHEQ1HqLYJnwx
  https://open.spotify.com/track/2UFEbNhbk9GsxCigXPq... → 2UFEbNhbk9GsxCigXPq3XI
  https://open.spotify.com/track/2HvTGx5fzFGpHSyRNvX... → 2HvTGx5fzFGpHSyRNvXd9T

🎵 Fetching track names and artists from Spotify

🔐 Connecting to Spotify...
✅ Connected

📊 Need to fetch metadata for 5014 tracks

🔍 Fetching in batches of 50...


Fetching:   0%|          | 0/101 [00:00<?, ?it/s]C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:108: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'Act Naturally - Remastered 2009' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'trackTitle'] = track_name
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_13444\1487600436.py:109: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'The Beatles' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  df.loc[idx, 'artists'] = artists
Fetching: 100%|██████████| 101/101 [01:07<00:00,  1.51it/s]


✅ Fetched 5014 track names and artists

📊 Checking for missing data...
Missing track titles: 7
Missing artists: 7

💾 Saving updated dataset...
✅ Saved to audio_features_with_names.csv
✅ Saved simplified version to track_audio_features.csv

📊 RESULTS

Total tracks: 5014
Tracks with names: 5007
Tracks with artists: 5007

🎵 Sample tracks with names and artists:
------------------------------------------------------------
 1. Act Naturally - Remastered 2009 - The Beatles
     Valence: 0.940, Energy: 0.447, Tempo: 93.0

 2. 1990 - Danse Mix - Jean Leloup
     Valence: 0.648, Energy: 0.939, Tempo: 121.9

 3. 12-Bar Original - Take 2 Edited - The Beatles
     Valence: 0.566, Energy: 0.535, Tempo: 122.7

 4. Across The Universe - Take 2 - The Beatles
     Valence: 0.463, Energy: 0.341, Tempo: 79.4

 5. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.840, Energy: 0.395, Tempo: 150.9

 6. Across The Universe - Remastered 2009 - The Beatles
     Valence: 0.858, Energy: 0.412,